In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import importlib

from kfirlib.comparisons import *
from kfirlib.util import *
from kfirlib import gpt
from pydantic import BaseModel

In [ ]:
ROOT = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\papers'

In [ ]:
pdf_paths = []
for current_dir, subdirs, files in os.walk(ROOT):
    for file in files:
        path = os.path.join(current_dir, file)
        if path.endswith('.pdf'):
            pdf_paths.append(path)

print(len(pdf_paths))
print(pdf_paths[0])

In [ ]:
client = gpt._CLIENT

In [ ]:
class SinglePdfResult(BaseModel):
    title: str
    answer: str

class AllResults(BaseModel):
    results: list[SinglePdfResult]

In [ ]:
files_with_paths = []
for path in tqdm(pdf_paths):
    files_with_paths.append({'path': path,
        'file_object': client.files.create(
        file=open(path, "rb"),
        purpose="user_data")})


In [ ]:
len(files_with_paths)

In [ ]:
system_prompt = '''
You are a scientist. You will be given several PDF files of scientific articles. You need to answer a question about all of them.

For each article, please output the article's title, then the answer. Your output should have the following format:

```
{
    "results": [
        {
            "title": "<article title>",
            "answer": "<answer to the question>",
        },
        {
            "title": "<article title>",
            "answer": "<answer to the question>",
        },
        ...
    ]
}

```


etc. 
You must output a result for **each** PDF.

Write out the results for all PDFs **in the same order as they are given**.
'''

In [ ]:
def run_model_on_batch(batch, question):
    content = [
                {
                    "type": "text",
                    "text": f"Please answer this question about each article:\n\n{question}",
                },]
    for item in batch:
        content.append({ 
            "type": "file", 
            "file": item['file_content'],
        })
    response = client.beta.chat.completions.parse(
        model=gpt.Models.FULL,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": content,
            }
        ],
        response_format=AllResults,
    )
    return response

In [ ]:
OUTPUTS = None
TO_PROCESS = None
QUESTION = "What are the health system barriers and enablers to HPV and MMR vaccination among underserved populations in high-income countries?"

In [ ]:
def run_on_missing_items(items, batch_size):
    global OUTPUTS, TO_PROCESS
    to_process = [item for item in items if 'response' not in item]
    TO_PROCESS = to_process
    print(f'Running on {len(to_process)} articles')

    outputs = []
    batches = list(batched(to_process, batch_size))
    for batch in tqdm(batches):
        outputs.append(run_model_on_batch(batch, QUESTION))

    OUTPUTS = outputs
    missing_items = 0
    updated_items = 0

    for i, (output, batch) in enumerate(zip(outputs, batches)):
        print(response)
        return
        try:
            res = parse_pdf_response(output)
        except Exception as e:
            print(f'Error in batch {i}: {e}')
            missing_items += len(batch)
            continue
        if len(res) != len(batch):
            print(f'Batch {i}: Expected {len(batch)} results but got {len(res)}.')
            missing_items += len(batch)
            continue
        for item, result in zip(batch, res):
            item.update(result)
        updated_items += len(batch)

    print(f'Updated {updated_items} items, missing {missing_items} items.')


In [ ]:
run_on_missing_items(files_with_paths, 6)

In [ ]:
run_on_missing_items(files_with_paths, 5)

In [ ]:
run_on_missing_items(files_with_paths, 3)

In [ ]:
run_on_missing_items(files_with_paths, 1)

In [ ]:
df = pd.DataFrame(files_with_paths)
del df['file_object']
df.rename(columns={'reasoning': 'full_article_reasoning', 'decision': 'full_article_decision'}, inplace=True)
df = df[~df.full_article_decision.isna()]
df.head()

In [ ]:
df.full_article_decision.sum()

In [ ]:
len(df), df.title.nunique()

In [ ]:
df.drop_duplicates('title', inplace=True)

In [ ]:
df['title_extract'] = df.title.apply(extract_letters)
len(df), df.title_extract.nunique()

In [ ]:
full_article_df = df

In [ ]:
old_df = pd.read_excel(r"C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\deduplicate_with_eligibility.xlsx")
old_df.head()

In [ ]:
old_df['title_extract'] = old_df.title.apply(extract_letters)

In [ ]:
#full_article_df = pd.read_excel(r"C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\combined_paul100_pdf_analysis.xlsx")

In [ ]:
combined_df = pd.merge(old_df, full_article_df, on='title_extract', how='outer')
len(combined_df)

In [ ]:
combined_df[['title_x', 'title_y']]

In [ ]:
combined_df.to_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\combined_deduplicated_pdf_analysis.xlsx', index=False)